# Sensitivity Landscape Workflow — alanine MVP

This notebook is a **thin viewer over the engine**: every real step calls the same
`src/` functions and `scripts/` the command line uses, so the notebook and CLI cannot
diverge. No analysis logic lives here — only calls into the engine plus display.

**The only thing you edit is the study config YAML** (Cell 1). Then choose **Run All**.
Outputs are written under `studies/<study_id>/`. With Cantera installed the notebook runs
end-to-end; without it, the run cell is skipped cleanly and the summary/plot/verdict cells
work over any existing results.

## Cell 1 — User setting (the only thing you edit)

In [ ]:
STUDY_CONFIG = "../studies/alanine_mvp/study_config.yaml"

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

# Robust project-root detection (works from notebooks/ or the repo root).
CWD0 = Path.cwd().resolve()
PROJECT_ROOT = CWD0.parent if CWD0.name == "notebooks" else CWD0
for sub in ("src", "scripts"):
    p = PROJECT_ROOT / sub
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Resolve the config path before chdir; then work from the repo root so the engine's
# relative paths (output_dir, species_files) resolve consistently.
_cfg = Path(STUDY_CONFIG)
CONFIG_PATH = _cfg if _cfg.is_absolute() else (CWD0 / _cfg).resolve()
os.chdir(PROJECT_ROOT)
print("Repo root:", PROJECT_ROOT)
print("Study config:", CONFIG_PATH)

## Cell 2 — Load & inspect the study\nValidates the config (plain-English error if invalid) and prints what will run.

In [ ]:
from sensitivity_design import (
    load_study_config, validate_study,
    load_species_for_config, load_gibbs_seed_for_config,
)

config = load_study_config(CONFIG_PATH)
species_df = load_species_for_config(config, PROJECT_ROOT)
gibbs_seed_df = load_gibbs_seed_for_config(config, PROJECT_ROOT)
validate_study(config, species_df, gibbs_seed_df)  # raises a plain-English error on bad input

study = config["study"]
enabled = [k for k, v in (config.get("sweeps") or {}).items() if v and v.get("enabled")]
print("study_id:          ", study["study_id"])
print("target products:   ", config.get("mode", {}).get("target_products"))
print("enabled substudies:", enabled)
print("allowed_species:   ", config.get("model", {}).get("allowed_species"))
print("thresholds:        ", config.get("thresholds"))
print("output_dir:        ", study.get("output_dir"))

## Cell 3 — Design preview (no run)\nThe in-notebook equivalent of `--dry-run`: case counts, ΔG variant names, grid ranges.

In [ ]:
from sensitivity_design import build_full_design_matrix, build_thermo_offsets_table

design = build_full_design_matrix(config)
print("Total cases:", len(design))
print(design["substudy_id"].value_counts().to_string())

offsets = build_thermo_offsets_table(config)
if not offsets.empty:
    print("\n%d ΔG variants:" % len(offsets), ", ".join(offsets["variant_species"]))

print("\nGrid ranges:")
for col in ["NH3_mol", "C2H2_mol", "C2H2_over_HCN", "deltaG_offset_kJ_mol"]:
    if col in design:
        print(f"  {col:<22} {design[col].min():>8.4g} .. {design[col].max():<8.4g}")

display(design.head(10))

## Cell 4 — Run the study\nCalls the exact CLI path (prepare coefficients → model/run manifests → equilibrium). Resume-by-default: only not-`ok` cases run. Needs Cantera; if it is missing the cell is skipped cleanly.

In [ ]:
import run_sensitivity_study

rc_run = run_sensitivity_study.main(["--config", str(CONFIG_PATH)])
if rc_run == 2:
    display(Markdown(
        "> **Cantera is not installed** — manifests/models were written, but the equilibrium "
        "run was skipped. The cells below summarize/plot existing results."))
elif rc_run != 0:
    raise RuntimeError(f"run_sensitivity_study exited with unexpected code {rc_run}.")

## Cell 5 — Summarize\nMole reconstruction → case summary → landscape grid → metrics → run-summary markdown + schema dictionary.

In [ ]:
import summarize_sensitivity_study

study_dir = PROJECT_ROOT / config["study"]["output_dir"]
results_dir = study_dir / "results"

rc_sum = summarize_sensitivity_study.main(["--config", str(CONFIG_PATH)])
if rc_sum != 0:
    display(Markdown("> No raw results yet — run the study (Cell 4) in a Cantera environment first."))
else:
    case_summary = pd.read_csv(results_dir / "sensitivity_case_summary.csv")
    n = len(case_summary)
    failed = int((case_summary["solver_status"] != "ok").sum())
    print("formation_call counts:", case_summary["formation_call"].value_counts().to_dict())
    print("suspect_balance cases:", int(case_summary["suspect_balance"].sum()))
    print(f"failure rate: {failed}/{n} ({failed / n:.1%})" if n else "no cases")
    display(Markdown((results_dir / "sensitivity_run_summary.md").read_text()))

## Cell 6 — Plots (inline)\nConfig-driven figures written by the engine; the cell only displays the PNGs.

In [ ]:
import plot_sensitivity_study

plot_sensitivity_study.main(["--config", str(CONFIG_PATH)])
figures_dir = study_dir / "figures"
for name in ["inventory_landscape", "inventory_landscape_autoscaled",
             "deltaG_sweep", "nh3_deltaG_landscape", "nh3_deltaG_landscape_autoscaled"]:
    png = figures_dir / f"{name}.png"
    if png.exists():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(png)))

## Cell 7 — MVP verdict\nReads the computed metrics (not hardcoded) via the tested `render_mvp_verdict` helper and states a scale / revise / pause recommendation.

In [ ]:
from sensitivity_summary import compute_sensitivity_metrics, render_mvp_verdict

summary_path = results_dir / "sensitivity_case_summary.csv"
if not summary_path.exists():
    display(Markdown("> No MVP verdict yet — run the study in a Cantera environment first, "
                     "or open a study folder that already has results."))
else:
    case_summary = pd.read_csv(summary_path)
    sig = float(config.get("thresholds", {}).get("significant_X_threshold", 1e-6))
    metrics = compute_sensitivity_metrics(case_summary, significant_X_threshold=sig)
    display(Markdown(render_mvp_verdict(metrics, config)))